# 2026-06-15 Day 3: Modern Decoder Block Upgrade Path

Goal: understand RoPE, RMSNorm, and SwiGLU well enough to plan the modern decoder-block upgrade without breaking the existing assignment.


## How To Use This Reading Notebook

Use this before touching implementation for the day. The goal is not to memorize the paper. The goal is to extract the model of the system you need for today's code and notes.

Workflow:

1. Read only the assigned sections.
2. Fill the mini-lecture answer in your own words.
3. Open the repo files listed in the roadmap.
4. Run the listed checks or record why they skip.
5. Write a short exit ticket before moving on.


## Assigned Reading

| Source | Exact Target | Why It Matters Today |
| --- | --- | --- |
| [LLaMA](https://arxiv.org/abs/2302.13971) | Section 2.2 | Shows modern block choices: pre-normalization, RMSNorm, SwiGLU, RoPE. |
| [RoFormer](https://arxiv.org/abs/2104.09864) | Sections 1-3 | Explains rotary position embeddings and why Q/K rotation carries relative-position information. |


## Reading Summary

### LLaMA Section 2.2

LLaMA follows the Transformer family but changes several block details that became common in modern decoder-only LMs. It uses pre-normalization, RMSNorm instead of LayerNorm, SwiGLU-style feed-forward networks, and rotary position embeddings. The project does not need LLaMA-scale training; it needs these architectural choices as implementation targets.

Key takeaway for implementation: each replacement should preserve the residual-stream contract `(B, T, C) -> (B, T, C)`.

### RoFormer Sections 1-3

RoFormer introduces rotary position embeddings. Instead of adding a position vector to hidden states, RoPE rotates query and key feature pairs by a position-dependent angle. The dot product between a rotated query and rotated key then depends on relative position. Values are not rotated because values are the content being mixed.

Key takeaway for implementation: RoPE belongs after Q/K projection and head splitting, before the attention score matrix.


## Diagram Anchor

![RoPE Q/K rotation](../../assets/figures/rope_qk_rotation.svg)

What to notice: Q and K get position-dependent rotations; V stays unchanged.

![RMSNorm versus LayerNorm](../../assets/figures/rmsnorm_vs_layernorm.svg)

What to notice: RMSNorm removes the mean-subtraction branch while preserving shape.

![SwiGLU MLP](../../assets/figures/swiglu_mlp.svg)

What to notice: the gate and value paths multiply before the output projection.


## Repo Connection

Open these files after reading:

| File | What To Find |
| --- | --- |
| `src/moe_transformer_lab/trainable/model.py` | Current `LayerNorm`, learned position embedding, GELU `DenseMLP`, and attention projections. |
| `src/moe_transformer_lab/config.py` | Fields that could later become opt-in architecture choices. |
| `tests/test_swappable_ffn.py` | Tests that should still pass after any modern-block additions. |


## Upgrade Map

| Current Repo Component | Modern Target | Shape Contract | Why It Matters |
| --- | --- | --- | --- |
| learned position embedding | RoPE | Q/K stay `(B,H,T,Dh)` | makes position enter attention scores |
| LayerNorm | RMSNorm | `(B,T,C) -> (B,T,C)` | simpler modern pre-norm choice |
| GELU MLP | SwiGLU | `(B,T,C) -> (B,T,C)` | gated FFN used by many LLMs |
| dense FFN or MoE | keep swappable interface | `(output, aux_loss)` | preserves MoE extension |


## Mini-Lecture Answer Seed

RoPE rotates Q and K, not V. The rotation happens after projection and head split, just before `QK^T`. RMSNorm rescales a token vector by its root-mean-square value but does not subtract the mean. SwiGLU uses two input projections: one value path and one gate path. The gate controls which expanded features pass through before the output projection returns to hidden width `C`.


## Active Recall

1. Where exactly should RoPE be applied?
2. Why is `Dh` usually required to be even for RoPE?
3. What does RMSNorm skip compared with LayerNorm?
4. Why can SwiGLU replace GELU MLP without changing the block interface?

## Exit Ticket

Write current block pseudocode and planned modern block pseudocode side by side.
